# Combined Feature Extraction Pipeline (All Families)

Builds a single dataset by merging every available extracted feature family. Feature columns are prefixed as `family__feature` to prevent collisions.

In [5]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'pyproject.toml').exists():
            return path
    raise FileNotFoundError('Could not locate repository root (missing pyproject.toml).')


REPO_ROOT = find_repo_root(Path.cwd())
PIPELINE_DIR = REPO_ROOT / 'feature_extraction' / 'pipelines'
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

from pipeline_common import (
    ANOVA_FAMILIES,
    DEFAULT_ALL_FAMILIES,
    METADATA_COLUMNS,
    feature_columns_from_families,
    load_base_metadata,
    load_family_frames,
    load_selected_feature_lists,
    merge_feature_families,
    select_prefixed_columns,
)

OUT_DIR = REPO_ROOT / 'extracted_features' / 'combined'
OUT_DIR.mkdir(parents=True, exist_ok=True)


In [6]:
FAMILIES = DEFAULT_ALL_FAMILIES.copy()

if AUTO_RUN_MISSING:
    gen_results = run_missing_family_generators(
        REPO_ROOT,
        families=FAMILIES,
        execute_timeout=NOTEBOOK_TIMEOUT_SECONDS,
    )
    if gen_results:
        display(pd.DataFrame(gen_results))


,family,csv_path,available,rows,features,status
1,mfcc_normalized,/home/marcello/Speech-Emotion-Recognition/extr...,True,7532,1150,ok
0,mfcc_raw,/home/marcello/Speech-Emotion-Recognition/extr...,True,7532,1150,ok
2,prosody_energy,/home/marcello/Speech-Emotion-Recognition/extr...,True,7532,8,ok
6,representations,/home/marcello/Speech-Emotion-Recognition/extr...,True,7532,136,ok
7,rhythm_pauses,/home/marcello/Speech-Emotion-Recognition/extr...,True,7532,10,ok
8,ssl_embeddings,/home/marcello/Speech-Emotion-Recognition/extr...,True,7532,1539,ok
5,bert,/home/marcello/Speech-Emotion-Recognition/extr...,False,0,0,missing_file
3,prosody_pitch,/home/marcello/Speech-Emotion-Recognition/extr...,False,0,0,missing_file
4,tfidf,/home/marcello/Speech-Emotion-Recognition/extr...,False,0,0,missing_file
9,tonality,/home/marcello/Speech-Emotion-Recognition/extr...,False,0,0,missing_file


Base rows: 7,532
Loaded families: 6 / 10


In [3]:
base_df = load_base_metadata(REPO_ROOT, include_xxx=True, require_agreement=True)
family_frames, family_report = load_family_frames(
    REPO_ROOT,
    families=FAMILIES,
    prefix_features=True,
)

report_df = pd.DataFrame(family_report).sort_values(['available', 'family'], ascending=[False, True])
display(report_df)

print(f'Base rows: {len(base_df):,}')
print(f'Loaded families: {len(family_frames):,} / {len(FAMILIES):,}')


,family,csv_path,available,rows,features,status,requested_features,missing_requested
5,bert,f:\Speech-Emotion-Recognition\extracted_featur...,True,7532,771,ok,None,None
1,mfcc_normalized,f:\Speech-Emotion-Recognition\extracted_featur...,True,7532,1150,ok,None,None
0,mfcc_raw,f:\Speech-Emotion-Recognition\extracted_featur...,True,7532,1150,ok,None,None
2,prosody_energy,f:\Speech-Emotion-Recognition\extracted_featur...,True,7532,8,ok,None,None
3,prosody_pitch,f:\Speech-Emotion-Recognition\extracted_featur...,True,7532,14,ok,None,None
6,representations,f:\Speech-Emotion-Recognition\extracted_featur...,True,7532,136,ok,None,None
7,rhythm_pauses,f:\Speech-Emotion-Recognition\extracted_featur...,True,7532,10,ok,None,None
8,ssl_embeddings,f:\Speech-Emotion-Recognition\extracted_featur...,True,7532,1539,ok,None,None
4,tfidf,f:\Speech-Emotion-Recognition\extracted_featur...,True,7532,9158,ok,None,None
9,tonality,f:\Speech-Emotion-Recognition\extracted_featur...,True,7532,91,ok,None,None


Base rows: 7,532
Loaded families: 10 / 10
Combined shape: (7532, 14034)


,path,session,method,gender,emotion,n_annotators,agreement,mfcc_raw__duration_s,mfcc_raw__mfcc13_mfcc_frames,mfcc_raw__mfcc13_mfcc_mean,...,tonality__tonnetz_04_mean,tonality__tonnetz_04_std,tonality__tonnetz_04_p10,tonality__tonnetz_04_p50,tonality__tonnetz_04_p90,tonality__tonnetz_05_mean,tonality__tonnetz_05_std,tonality__tonnetz_05_p10,tonality__tonnetz_05_p50,tonality__tonnetz_05_p90
0,Session1/sentences/wav/Ses01F_script02_1/Ses01...,1,script,F,neu,3,3,2.070000,130.0,-25.572800,...,0.002719,0.030271,-0.033143,-0.005690,0.041914,-0.029146,0.024235,-0.059114,-0.025230,-0.004007
1,Session1/sentences/wav/Ses01F_script02_1/Ses01...,1,script,F,fru,3,2,1.502438,94.0,-27.364491,...,0.000670,0.012330,-0.015786,0.002184,0.016823,-0.027483,0.013177,-0.041836,-0.028746,-0.009464


In [7]:
merged_df = merge_feature_families(base_df, family_frames)
feature_cols = feature_columns_from_families(merged_df, list(family_frames.keys()))

all_nan_cols = [c for c in feature_cols if merged_df[c].isna().all()]
if all_nan_cols:
    merged_df = merged_df.drop(columns=all_nan_cols)
    feature_cols = [c for c in feature_cols if c not in all_nan_cols]
    print(f'Dropped all-NaN feature columns: {len(all_nan_cols):,}')

meta_cols = [c for c in METADATA_COLUMNS if c in merged_df.columns]
out_cols = [*meta_cols, *feature_cols]
combined_df = merged_df[out_cols].copy()

print(f'Combined shape: {combined_df.shape}')
print(f'Metadata columns: {len(meta_cols)} | Feature columns: {len(feature_cols):,}')

combined_df.head(2)


Combined shape: (7532, 4000)
Metadata columns: 7 | Feature columns: 3,993


,path,session,method,gender,emotion,n_annotators,agreement,mfcc_raw__duration_s,mfcc_raw__mfcc13_mfcc_frames,mfcc_raw__mfcc13_mfcc_mean,...,ssl_embeddings__ssl_std_0758,ssl_embeddings__ssl_std_0759,ssl_embeddings__ssl_std_0760,ssl_embeddings__ssl_std_0761,ssl_embeddings__ssl_std_0762,ssl_embeddings__ssl_std_0763,ssl_embeddings__ssl_std_0764,ssl_embeddings__ssl_std_0765,ssl_embeddings__ssl_std_0766,ssl_embeddings__ssl_std_0767
0,Session1/sentences/wav/Ses01F_script02_1/Ses01...,1,script,F,neu,3,3,2.070000,130.0,-25.572798,...,0.191622,0.315093,0.105752,0.181741,0.151031,0.081244,0.064333,0.105190,0.119758,0.325718
1,Session1/sentences/wav/Ses01F_script02_1/Ses01...,1,script,F,fru,3,2,1.502438,94.0,-27.364491,...,0.133078,0.256237,0.104170,0.151399,0.071512,0.073623,0.059866,0.100028,0.082334,0.350629


In [8]:
OUT_CSV = OUT_DIR / 'all_families_features.csv'
combined_df.to_csv(OUT_CSV, index=False)
print(f'Saved: {OUT_CSV}')


Saved: /home/marcello/Speech-Emotion-Recognition/extracted_features/combined/all_families_features.csv
